# Lesson 2b: Multilayer Perceptrons and Backpropagation — Practical

The companion theory notebook (2a) derived backpropagation from the chain
rule, implemented the forward and backward passes **from scratch in
NumPy** for an `[784, 64, 32, 10]` MLP, verified the analytic gradients
against a numerical finite-difference check, and trained that hand-rolled
network on a real MNIST subset.

This notebook reproduces the same model — same architecture, same weight
initialisation, same loss — with the production framework (PyTorch), and
answers the question 2a's closing note raised: does autograd's automatic
backward pass agree with the hand-derived one, parameter for parameter?

By the end of this notebook you will have:
- rebuilt 2a's exact `[784, 64, 32, 10]` ReLU MLP as an `nn.Module`,
- confirmed that **autograd's gradients match the manual/analytic NumPy
  gradients** to numerical tolerance, on identical weights and identical
  input,
- registered a **backward hook** on every layer to record per-layer
  gradient magnitudes during training, and plotted how they evolve, and
- trained the PyTorch MLP on an MNIST subset and reported test accuracy.

## Introduction

2a's argument was mathematical and mechanical: derive
$\delta^{[L]} = a^{[L]} - y$ and $\delta^{[l]} = (W^{[l+1]})^\top
\delta^{[l+1]} \odot g'(z^{[l]})$ from the chain rule, then implement those
boxed formulas by hand in a NumPy `MLP` class with no autograd anywhere.
That from-scratch backward pass was checked two ways: against a numerical
finite-difference gradient on a toy network, and by actually training on
MNIST and watching the loss fall.

In practice, nobody hand-derives backward passes for real networks —
frameworks like PyTorch compute them automatically via **autograd**, which
builds a computation graph during the forward pass and walks it backward,
applying the chain rule at every node exactly as 2a did by hand. This
notebook makes that equivalence concrete rather than assumed: we build the
identical architecture in PyTorch, copy 2a's NumPy weights into it exactly,
run one forward-and-backward pass on the same batch in both frameworks, and
compare the resulting gradients directly. If autograd is doing what 2a's
derivation says it should, the two sets of gradients must agree to
numerical precision.

We then go one step further than 2a did: a **backward hook** attached to
every layer lets us *watch* the error signal $\delta^{[l]}$ (and hence the
weight gradient magnitude) at each layer during real training on MNIST —
a first, concrete look at how gradient magnitude varies with depth, which
becomes central when Lesson 3 discusses vanishing and exploding
gradients.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, data subsampling) is reproducible. Seed both numpy and torch, and do
# it before anything random happens.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

### Device check

This notebook is written to run unmodified in Google Colab or locally. If a
GPU is available (as it typically is on a Colab GPU runtime) we use it;
otherwise we fall back to CPU. Every tensor and module below is moved to
`device` explicitly.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

### Loading MNIST

`torchvision.datasets.MNIST` downloads the dataset automatically the first
time this cell runs, to a local `data/` folder next to this notebook — this
works identically in Colab and locally, no manual setup required. We
subsample heavily to keep the whole notebook well under the runtime budget
on a CPU, exactly as 2a did — a couple thousand training images and a
matching held-out test set, teaching example not leaderboard run.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_full = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_full = torchvision.datasets.MNIST(root="data", train=False, download=True, transform=transform)


def subsample(dataset, n_total, seed):
    g = np.random.default_rng(seed)
    idx = g.permutation(len(dataset))[:n_total]
    images = torch.stack([dataset[i][0] for i in idx])           # (n, 1, 28, 28)
    X = images.reshape(len(idx), -1).numpy().astype(np.float64)  # rows = examples, matches sklearn/PyTorch convention
    labels = dataset.targets.numpy()[idx].astype(int)
    return X, labels


N_TRAIN, N_TEST = 2000, 400
X_train, labels_train = subsample(train_full, N_TRAIN, seed=SEED)
X_test, labels_test = subsample(test_full, N_TEST, seed=SEED + 1)

print("X_train:", X_train.shape, " labels_train:", labels_train.shape)
print("X_test: ", X_test.shape, " labels_test: ", labels_test.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"label: {labels_train[i]}", fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST training samples")
plt.show()

## The PyTorch MLP

2a's from-scratch `MLP` class hard-coded `layer_sizes=[784, 64, 32,
10]` with ReLU hidden activations, He-initialised weights, and a softmax
output layer trained with categorical cross-entropy. Every one of those
choices maps directly onto PyTorch building blocks:

| 2a (NumPy, from scratch)                  | PyTorch equivalent                         |
|--------------------------------------------|---------------------------------------------|
| `W[l] @ a + b[l]`, He-initialised           | `nn.Linear(in, out)` per layer              |
| `relu(z)` on hidden layers                  | `nn.ReLU()` between `nn.Linear` layers      |
| `softmax(z)` + categorical cross-entropy    | raw logits + `nn.CrossEntropyLoss` (applies log-softmax + NLL in one numerically stable op) |
| manual `.backward()`                        | autograd (`loss.backward()`)                |

Using `nn.CrossEntropyLoss` on raw logits (rather than a separate
`nn.Softmax` + NLL loss) is the PyTorch idiom for exactly the same reason
2a's derivation found the $\delta^{[L]} = a^{[L]} - y$ cancellation clean:
combining softmax and cross-entropy avoids computing `log(softmax(z))`
element-by-element, which is numerically fragile for large $|z|$.

In [ ]:
class TorchMLP(nn.Module):
    '''Same architecture as 2a's from-scratch MLP: [784, 64, 32, 10],
    ReLU hidden activations, linear output (softmax is folded into the loss).'''

    def __init__(self, layer_sizes=(784, 64, 32, 10)):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 2):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # raw logits; nn.CrossEntropyLoss applies log-softmax internally


torch.manual_seed(SEED)
model = TorchMLP().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"total parameters: {n_params}")

## Autograd vs Manual Gradients

To confirm autograd genuinely reproduces 2a's hand-derived backward
pass — not merely "looks similar", but agrees to numerical precision — we
rebuild 2a's exact NumPy `MLP` here, copy its He-initialised weights
**directly into** the PyTorch model's parameters (so both networks start
from bit-for-bit identical weights), run one forward-and-backward pass on
the *same* input batch in both frameworks, and compare every resulting
weight and bias gradient array elementwise.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def relu_deriv(z):
    return (z > 0).astype(z.dtype)


def softmax(z):
    z = z - z.max(axis=0, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=0, keepdims=True)


class NumpyMLP:
    '''2a's from-scratch MLP: hand-derived forward/backward, no autograd.
    Data convention: examples stacked as columns, shape (n_features, n_examples).'''

    def __init__(self, layer_sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.L = len(layer_sizes) - 1
        self.W, self.b = [], []
        for l in range(self.L):
            fan_in, fan_out = layer_sizes[l], layer_sizes[l + 1]
            scale = np.sqrt(2.0 / fan_in)  # He initialization
            self.W.append(rng.normal(scale=scale, size=(fan_out, fan_in)))
            self.b.append(np.zeros((fan_out, 1)))

    def forward(self, X):
        self.cache = {"a0": X}
        a = X
        for l in range(1, self.L + 1):
            z = self.W[l - 1] @ a + self.b[l - 1]
            a = softmax(z) if l == self.L else relu(z)
            self.cache[f"z{l}"] = z
            self.cache[f"a{l}"] = a
        return a

    def backward(self, Y):
        n = Y.shape[1]
        grads_W, grads_b = [None] * self.L, [None] * self.L
        delta = self.cache[f"a{self.L}"] - Y  # softmax + cross-entropy cancellation
        for l in range(self.L, 0, -1):
            a_prev = self.cache[f"a{l - 1}"]
            grads_W[l - 1] = (delta @ a_prev.T) / n
            grads_b[l - 1] = delta.sum(axis=1, keepdims=True) / n
            if l > 1:
                z_prev = self.cache[f"z{l - 1}"]
                delta = (self.W[l - 1].T @ delta) * relu_deriv(z_prev)
        return grads_W, grads_b


LAYER_SIZES = [784, 64, 32, 10]
numpy_model = NumpyMLP(LAYER_SIZES, seed=SEED)
print("NumPy MLP built with He-initialised weights:",
      [w.shape for w in numpy_model.W])

In [ ]:
# Copy the NumPy model's weights into the PyTorch model so both networks
# start from bit-for-bit identical parameters. nn.Linear stores weight as
# (out_features, in_features) — the same convention as NumpyMLP's W[l].
with torch.no_grad():
    linear_layers = [m for m in model.net if isinstance(m, nn.Linear)]
    assert len(linear_layers) == numpy_model.L
    for layer, W, b in zip(linear_layers, numpy_model.W, numpy_model.b):
        layer.weight.copy_(torch.tensor(W, dtype=torch.float32))
        layer.bias.copy_(torch.tensor(b.flatten(), dtype=torch.float32))

print("Copied NumPy weights into the PyTorch model — both networks now start identically.")

In [ ]:
# One shared batch, same input and labels for both frameworks.
BATCH_N = 64
batch_idx = np.random.default_rng(SEED).choice(N_TRAIN, size=BATCH_N, replace=False)
X_batch = X_train[batch_idx]              # (BATCH_N, 784), rows = examples
labels_batch = labels_train[batch_idx]    # (BATCH_N,)

# --- NumPy forward + backward (2a's convention: columns = examples) ---
X_batch_cols = X_batch.T                                    # (784, BATCH_N)
Y_batch_onehot = np.zeros((10, BATCH_N))
Y_batch_onehot[labels_batch, np.arange(BATCH_N)] = 1.0

numpy_model.forward(X_batch_cols)
grads_W_numpy, grads_b_numpy = numpy_model.backward(Y_batch_onehot)

# --- PyTorch forward + backward (rows = examples, autograd computes gradients) ---
model.zero_grad()
x_t = torch.tensor(X_batch, dtype=torch.float32).to(device)
y_t = torch.tensor(labels_batch, dtype=torch.long).to(device)
logits = model(x_t)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, y_t)
loss.backward()

grads_W_torch = [layer.weight.grad.detach().cpu().numpy() for layer in linear_layers]
grads_b_torch = [layer.bias.grad.detach().cpu().numpy() for layer in linear_layers]

print(f"NumPy loss (categorical cross-entropy): {-np.mean(np.sum(Y_batch_onehot * np.log(numpy_model.cache[f'a{numpy_model.L}'] + 1e-12), axis=0)):.6f}")
print(f"PyTorch loss (CrossEntropyLoss):        {loss.item():.6f}")

If autograd truly reproduces the hand-derived backward pass, the
weight and bias gradients computed by `.backward()` (NumPy, hand-derived)
and by `loss.backward()` (PyTorch, autograd) should agree to floating-point
precision — the only source of disagreement should be the `float64` (NumPy)
vs. `float32` (PyTorch) precision difference, not any structural mismatch.

In [ ]:
print(f"{'layer':<8}{'max abs diff (W)':>18}{'max rel diff (W)':>18}{'max abs diff (b)':>18}{'max rel diff (b)':>18}")
max_rel_overall = 0.0
for l, (gW_np, gW_t, gb_np, gb_t) in enumerate(zip(grads_W_numpy, grads_W_torch, grads_b_numpy, grads_b_torch)):
    gb_np_flat = gb_np.flatten()

    abs_diff_W = np.abs(gW_np - gW_t)
    rel_diff_W = abs_diff_W / (np.abs(gW_np) + np.abs(gW_t) + 1e-8)
    abs_diff_b = np.abs(gb_np_flat - gb_t)
    rel_diff_b = abs_diff_b / (np.abs(gb_np_flat) + np.abs(gb_t) + 1e-8)

    max_rel_overall = max(max_rel_overall, rel_diff_W.max(), rel_diff_b.max())
    print(f"{l:<8}{abs_diff_W.max():>18.2e}{rel_diff_W.max():>18.2e}"
          f"{abs_diff_b.max():>18.2e}{rel_diff_b.max():>18.2e}")

print(f"\nmax relative difference across every layer's W and b: {max_rel_overall:.2e}")
assert max_rel_overall < 1e-3, "autograd gradients should match the hand-derived NumPy gradients closely"
print("PASS: autograd's gradients match 2a's hand-derived analytic gradients to numerical tolerance.")

The two gradients agree to well within floating-point tolerance
(the residual difference is explained entirely by `float32` vs. `float64`
precision) — direct, numerical confirmation that PyTorch's autograd is
computing *exactly* the chain-rule recursion 2a derived and implemented by
hand: $\delta^{[L]} = a^{[L]} - y$, then $\delta^{[l]} =
(W^{[l+1]})^\top\delta^{[l+1]} \odot g'(z^{[l]})$, then $\nabla_{W^{[l]}}J =
\delta^{[l]}(a^{[l-1]})^\top$. Autograd is not a different algorithm from
manual backpropagation — it is the same algorithm, applied automatically by
walking the computation graph built during the forward pass.

## Inspecting Gradients with Hooks

Autograd normally discards intermediate gradients once they have
been used to populate `.grad` on leaf parameters — the internal
$\delta^{[l]}$ values 2a's derivation named explicitly are not retained by
default. A **backward hook**, registered with `register_full_backward_hook`,
is called during the backward pass with the gradient flowing *into* and
*out of* a module, letting us intercept and record exactly those
intermediate error signals without changing training behaviour at all.

We attach one hook per `nn.Linear` layer and record the mean absolute
gradient of that layer's output (i.e. $\text{mean}|\delta^{[l]}|$) on every
training step, giving a direct, layer-by-layer view of how strongly error
signal is flowing backward through the network — the same quantity 2a's
recursive formula computes, now observed from live training rather than
derived on paper. This is exactly the diagnostic Lesson 3 will reuse to
study vanishing and exploding gradients in deeper networks.

In [ ]:
grad_magnitude_history = {i: [] for i in range(len(linear_layers))}


def make_hook(layer_idx):
    def hook(module, grad_input, grad_output):
        # grad_output[0] is d(loss)/d(module output) — exactly delta^[l] for this layer.
        grad_magnitude_history[layer_idx].append(grad_output[0].abs().mean().item())
    return hook


hook_handles = [layer.register_full_backward_hook(make_hook(i))
                for i, layer in enumerate(linear_layers)]
print(f"registered backward hooks on {len(hook_handles)} Linear layers")

### Training with hooks attached

We now train the PyTorch MLP for real — mini-batch SGD on the MNIST
subset — with the hooks above firing on every `.backward()` call, silently
accumulating per-layer gradient magnitude history alongside normal
training.

In [ ]:
def accuracy(model, X, labels, device):
    model.eval()
    with torch.no_grad():
        x_t = torch.tensor(X, dtype=torch.float32).to(device)
        preds = model(x_t).argmax(dim=1).cpu().numpy()
    return (preds == labels).mean()


Xtr_t = torch.tensor(X_train, dtype=torch.float32)
ytr_t = torch.tensor(labels_train, dtype=torch.long)
train_dataset = TensorDataset(Xtr_t, ytr_t)

BATCH_SIZE = 64
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)

# Fresh model for the real training run (the weight-copied model above was
# used only for the one-shot gradient comparison and must not be reused here
# — reusing it would mean training resumes from a batch already stepped on).
torch.manual_seed(SEED)
train_model = TorchMLP().to(device)
linear_layers_train = [m for m in train_model.net if isinstance(m, nn.Linear)]
grad_magnitude_history = {i: [] for i in range(len(linear_layers_train))}
hook_handles = [layer.register_full_backward_hook(make_hook(i))
                for i, layer in enumerate(linear_layers_train)]

optimizer = torch.optim.SGD(train_model.parameters(), lr=0.3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 12
history = {"train_loss": [], "train_acc": [], "test_acc": []}
for epoch in range(EPOCHS):
    train_model.train()
    epoch_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = train_model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    history["train_loss"].append(np.mean(epoch_losses))
    history["train_acc"].append(accuracy(train_model, X_train, labels_train, device))
    history["test_acc"].append(accuracy(train_model, X_test, labels_test, device))

for handle in hook_handles:
    handle.remove()

print(f"final train loss: {history['train_loss'][-1]:.4f}")
print(f"final train accuracy: {history['train_acc'][-1]:.3f}")
print(f"final test accuracy:  {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"])
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training cross-entropy loss")
axes[0].set_title("PyTorch MLP: training loss")
axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="train accuracy")
axes[1].plot(history["test_acc"], label="test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title("PyTorch MLP: accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

assert history["train_loss"][-1] < history["train_loss"][0] * 0.5, "loss should drop substantially"
assert history["test_acc"][-1] > 0.7, "a correctly-trained MLP should clear 70% on this MNIST subset"
print("Confirmed: the PyTorch MLP trains successfully — loss falls sharply, "
      "test accuracy climbs well above the 10% chance level.")

### Per-layer gradient magnitude over training

Each hook recorded one value per mini-batch step (not per epoch), so
plotting the raw history shows how $\text{mean}|\delta^{[l]}|$ evolves
step by step across the whole training run, for every layer simultaneously.
`linear_layers_train[0]` is the first hidden layer (closest to the input,
layer index 0 in the `Sequential`), and the last entry is the output
layer.

In [ ]:
plt.figure(figsize=(8, 4.5))
layer_labels = ["layer 1 (784->64)", "layer 2 (64->32)", "layer 3 (32->10, output)"]
for i, label in zip(range(len(linear_layers_train)), layer_labels):
    plt.plot(grad_magnitude_history[i], label=label, alpha=0.8)
plt.xlabel("training step (mini-batch)")
plt.ylabel("mean |gradient| flowing into layer output")
plt.title("Per-layer gradient magnitude during training (via backward hooks)")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("mean gradient magnitude per layer, averaged over the full training run:")
for i, label in zip(range(len(linear_layers_train)), layer_labels):
    values = np.array(grad_magnitude_history[i])
    print(f"  {label:<24s}: mean {values.mean():.4e}, min {values.min():.4e}, max {values.max():.4e}")

The output layer's gradient starts largest — it receives
$\delta^{[L]} = a^{[L]} - y$ directly, with no intervening layer to shrink
it — while the earliest hidden layer's gradient has already been through
one backward step of $(W^{[l+1]})^\top\delta^{[l+1]} \odot g'(z^{[l]})$
before we observe it. For a 3-layer ReLU network this shallow, magnitudes
stay within the same order of magnitude across layers; Lesson 3 revisits
this exact hook-based diagnostic on deeper networks, where the repeated
multiplication in that recursion can make gradient magnitude shrink (or
grow) by orders of magnitude with depth.

## Key Takeaways

- The PyTorch `nn.Module` rebuild of 2a's `[784, 64, 32, 10]`
  from-scratch MLP is a direct, layer-for-layer translation:
  `nn.Linear` + `nn.ReLU()` per hidden layer, raw logits into
  `nn.CrossEntropyLoss` in place of a separate softmax + cross-entropy pair.
- With **identical weights and an identical input batch**, autograd's
  `loss.backward()` gradients matched 2a's hand-derived analytic gradients
  to within floating-point tolerance, for every layer's weights and
  biases — direct numerical confirmation that autograd implements the same
  chain-rule recursion 2a derived by hand, not a different algorithm.
- A **`register_full_backward_hook`** on each `nn.Linear` layer intercepts
  the gradient flowing through that layer during `.backward()` — exactly
  the $\delta^{[l]}$ error signal from 2a's derivation — without changing
  training behaviour, letting us observe per-layer gradient magnitude live
  during a real training run.
- The trained PyTorch MLP reached test accuracy well above the 10% chance
  level on the MNIST subset, and the per-layer gradient-magnitude plot
  showed magnitudes varying by layer depth even in this shallow 3-layer
  network — the same quantity, and the same hook technique, Lesson 3 will
  reuse to diagnose vanishing and exploding gradients in deeper networks.
- Practically: whenever a hand-derived gradient needs checking against a
  framework implementation, the recipe is always the same — copy identical
  weights into both, run one forward/backward pass on identical data, and
  compare gradient arrays directly. This is the same discipline as 2a's
  finite-difference gradient check, applied framework-to-framework instead
  of analytic-to-numerical.